<a href="https://colab.research.google.com/github/divya-dataengineer/data-engineer-playbook/blob/main/error_handlings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Error_Handling").getOrCreate()

data = [(1, "Divya", 23, 50000), (2, "Keerthi", 28, 40000), (3, "Uday", 28, 100000), (4, "Kumar", 34, 60000)]
columns = ["id", "name", "age", "salary"]

df = spark.createDataFrame(data, columns)
df.show(truncate=False)

+---+-------+---+------+
|id |name   |age|salary|
+---+-------+---+------+
|1  |Divya  |23 |50000 |
|2  |Keerthi|28 |40000 |
|3  |Uday   |28 |100000|
|4  |Kumar  |34 |60000 |
+---+-------+---+------+



In [6]:
from pyspark.sql.functions import lit

df.show()

#The error occurs because the withColumn function requires a second argument,
#which is the column expression that defines the values for the new column.
#You've provided the new column name, "result", but not the expression.
#👉 Always remember:
#column name + value/expression

TypeError: DataFrame.withColumn() missing 1 required positional argument: 'col'

In [7]:
#corrected answer
from pyspark.sql.functions import lit

df = df.withColumn("result", lit("pass"))
df.show(truncate=False)


+---+-------+---+------+------+
|id |name   |age|salary|result|
+---+-------+---+------+------+
|1  |Divya  |23 |50000 |pass  |
|2  |Keerthi|28 |40000 |pass  |
|3  |Uday   |28 |100000|pass  |
|4  |Kumar  |34 |60000 |pass  |
+---+-------+---+------+------+



In [11]:
#2. Using Python Operators Instead of PySpark Functions

df = df.filter(df.salary > 50000 or df.age > 25)
df.show()

PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.

In [10]:
#corrected answer

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Error_Handling").getOrCreate()

data = [(1, "Divya", 23, 50000), (2, "Keerthi", 28, 40000), (3, "Uday", 28, 100000), (4, "Kumar", 34, 60000)]
columns = ["id", "name", "age", "salary"]

df = spark.createDataFrame(data, columns)
df.show(truncate=False)

df = df.filter((col("salary") > 50000) | (col("age") > 25))
df.show(truncate=False)



+---+-------+---+------+
|id |name   |age|salary|
+---+-------+---+------+
|1  |Divya  |23 |50000 |
|2  |Keerthi|28 |40000 |
|3  |Uday   |28 |100000|
|4  |Kumar  |34 |60000 |
+---+-------+---+------+

+---+-------+---+------+
|id |name   |age|salary|
+---+-------+---+------+
|2  |Keerthi|28 |40000 |
|3  |Uday   |28 |100000|
|4  |Kumar  |34 |60000 |
+---+-------+---+------+



In [15]:
#3. Forgetting Immutability

df.withColumn("Result", col("Salary") * 0.1)
df.show(truncate=False)

#without assigning the data frame (df =) and result is geeting from the previous dataframe.

+---+-------+---+------+-------+
|id |name   |age|salary|Result |
+---+-------+---+------+-------+
|2  |Keerthi|28 |40000 |4000.0 |
|3  |Uday   |28 |100000|10000.0|
|4  |Kumar  |34 |60000 |6000.0 |
+---+-------+---+------+-------+



In [14]:
#corrected answer

df = df.withColumn("Result", col("Salary") * 0.1)
df.show(truncate=False)

#updated dataframe is coming as per the logic after assgned it to df =


+---+-------+---+------+-------+
|id |name   |age|salary|Result |
+---+-------+---+------+-------+
|2  |Keerthi|28 |40000 |4000.0 |
|3  |Uday   |28 |100000|10000.0|
|4  |Kumar  |34 |60000 |6000.0 |
+---+-------+---+------+-------+



In [16]:
#4 Using = Instead of == in Filters

df.filter(col("name") = "Divya")
df.show(truncate=False)


SyntaxError: expression cannot contain assignment, perhaps you meant "=="? (1990667899.py, line 3)

In [20]:
#corrected answer

df.filter(col("name") == "Keerthi").show(truncate=False)

+---+-------+---+------+------+
|id |name   |age|salary|Result|
+---+-------+---+------+------+
|2  |Keerthi|28 |40000 |4000.0|
+---+-------+---+------+------+



In [21]:
#5. Not Using col() for Column Reference
#Works sometimes but not recommended

df.filter(df.salary > 50000).show(truncate=False)

+---+-----+---+------+-------+
|id |name |age|salary|Result |
+---+-----+---+------+-------+
|3  |Uday |28 |100000|10000.0|
|4  |Kumar|34 |60000 |6000.0 |
+---+-----+---+------+-------+



In [22]:
#corrected answer

from pyspark.sql.functions import col

df.filter(col("salary") >5000).show(truncate=False)

+---+-------+---+------+-------+
|id |name   |age|salary|Result |
+---+-------+---+------+-------+
|2  |Keerthi|28 |40000 |4000.0 |
|3  |Uday   |28 |100000|10000.0|
|4  |Kumar  |34 |60000 |6000.0 |
+---+-------+---+------+-------+



In [37]:
#6. Forgetting Broadcast Join
#Causes performance issue

df1 = spark.createDataFrame([(1, "Divya"), (2, "Keerthi")], ["id", "name"])
df2 = spark.createDataFrame([(1, "Uday")], ["id", "name"])

df1.join(df2, "id").show()
#

+---+-----+----+
| id| name|name|
+---+-----+----+
|  1|Divya|Uday|
+---+-----+----+



In [36]:
#corrected answer

from pyspark.sql.functions import broadcast

df1 = spark.createDataFrame([(1, "Divya"), (2, "Keerthi")], ["id", "name"])
df2 = spark.createDataFrame([(1, "Uday")], ["id", "name"])

df1.join(broadcast(df2), "id").show()

+---+-----+----+
| id| name|name|
+---+-----+----+
|  1|Divya|Uday|
+---+-----+----+

